# 05 — So sánh unsupervised vs supervised

> **Đầu việc 05/6** trong pipeline phát hiện xâm nhập NSL-KDD.

**Mục tiêu**
- Huấn luyện baseline có giám sát: Logistic Regression, Random Forest, XGBoost.
- Dùng cùng tập train/test và cùng bộ metric với phương pháp unsupervised.
- So sánh công bằng, ghi rõ khác biệt về dữ liệu huấn luyện.

**Sản phẩm**: bảng metric tổng hợp, recall theo họ tấn công.

## 0. Môi trường

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nids.config import (CATEGORICAL, DATA_PROCESSED, FEATURES, RANDOM_STATE,
                         REPORTS, FAMILY_ORDER)
from nids.data import load_train_test, to_binary_label, to_family_label, novel_attack_types

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
np.random.seed(RANDOM_STATE)
print("Project root:", PROJECT_ROOT)
print("RANDOM_STATE =", RANDOM_STATE)

Project root: /mnt/Data/ptit/Network_Intrusion&Cyber_Attack_Detection
RANDOM_STATE = 36


## 1. Thiết kế so sánh

- Unsupervised: chỉ học từ Normal (hoặc toàn bộ train không nhãn) → score từ notebook 03/04.
- Supervised: học từ toàn bộ train có nhãn `Normal`/`Attack`.
- Cùng test set, cùng ngưỡng quyết định được báo cáo.

In [2]:
# 1) Nạp dữ liệu từ notebook 01
X_train = np.load(DATA_PROCESSED / "X_train.npy")
X_test = np.load(DATA_PROCESSED / "X_test.npy")
y_train_binary = np.load(DATA_PROCESSED / "y_train_binary.npy", allow_pickle=True)
y_test_binary = np.load(DATA_PROCESSED / "y_test_binary.npy", allow_pickle=True)
print("train:", X_train.shape, "test:", X_test.shape)

# Nhãn unsupervised đã chốt ở notebook 04
y_pred_euc = np.load(DATA_PROCESSED / "y_pred_euclidean.npy", allow_pickle=True)
y_pred_mah = np.load(DATA_PROCESSED / "y_pred_mahalanobis.npy", allow_pickle=True)

train: (125973, 121) test: (22544, 121)


## 2. Baseline có giám sát

In [3]:
# 2) Huấn luyện baseline supervised trên TOÀN BỘ train có nhãn
#    Cả 3 mô hình đều dùng cùng tập train/test với unsupervised.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# LogisticRegression: baseline tuyến tính
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train, y_train_binary)
y_pred_logreg = logreg.predict(X_test)

# RandomForest: model cây, bắt phi tuyến
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train_binary)
y_pred_rf = rf.predict(X_test)

# XGBoost: gradient boosting, thường mạnh nhất trong 3 baseline
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    subsample=0.9, colsample_bytree=0.9,
                    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
xgb.fit(X_train, y_train_binary)
y_pred_xgb = xgb.predict(X_test)

print("đã fit xong 3 model supervised")

đã fit xong 3 model supervised


## 3. Điểm unsupervised

Tái sử dụng score/ngưỡng đã chốt ở notebook 04.

In [4]:
# 3) Điểm unsupervised (ngưỡng đã chốt ở notebook 04)
# Flight: chỉ dùng Normal để ước lượng phân phối, ngưỡng 95th trên train-Normal.
# Không train thêm gì mới ở đây, chỉ nạp lại nhãn prediction.
y_pred_unsup_euclid = y_pred_euc
y_pred_unsup_mah = y_pred_mah
print("unsupervised predictions đã sẵn sàng")

unsupervised predictions đã sẵn sàng


## 4. Bảng metric chung

Cùng các cột: precision, recall, F1, PR-AUC.

In [5]:
# 4) Bảng metric chung cho mọi mô hình
from sklearn.metrics import precision_recall_fscore_support, average_precision_score

def row(y_true, y_pred):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0)
    return {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)}

# PR-AUC cần score liên tục; với supervised dùng xác suất predict_proba,
# với unsupervised dùng anomaly score (lấy từ notebook 03).
score_euc_test = np.load(DATA_PROCESSED / "score_euclidean_test.npy")
score_mah_test = np.load(DATA_PROCESSED / "score_mahalanobis_test.npy")

models = {
    "LogReg (supervised)": (y_pred_logreg, logreg.predict_proba(X_test)[:, 1]),
    "RandomForest (supervised)": (y_pred_rf, rf.predict_proba(X_test)[:, 1]),
    "XGBoost (supervised)": (y_pred_xgb, xgb.predict_proba(X_test)[:, 1]),
    "Euclidean (unsupervised)": (y_pred_unsup_euclid, score_euc_test),
    "Mahalanobis (unsupervised)": (y_pred_unsup_mah, score_mah_test),
}

rows = {}
for name, (pred, score) in models.items():
    r = row(y_test_binary, pred)
    r["pr_auc"] = round(float(average_precision_score(y_test_binary, score)), 4)
    rows[name] = r

table = pd.DataFrame(rows).T
table

,precision,recall,f1,pr_auc
LogReg (supervised),0.9176,0.6232,0.7423,0.8646
RandomForest (supervised),0.9678,0.6160,0.7528,0.9636
XGBoost (supervised),0.9681,0.6428,0.7726,0.9721
Euclidean (unsupervised),0.8836,0.4352,0.5832,0.8839
Mahalanobis (unsupervised),0.9370,0.8334,0.8822,0.9616


## 5. Recall theo họ tấn công

Tách DoS / Probe / R2L / U2R để thấy mô hình nào bỏ sót họ hiếm.

In [6]:
# 5) Recall theo họ tấn công (DoS/Probe/R2L/U2R) + Normal
from nids.data import to_family_label

train_raw, test_raw = load_train_test()
y_test_family = to_family_label(test_raw).to_numpy()

families = ["DoS", "Probe", "R2L", "U2R"]
recall_by_family = {}
for name, (pred, _) in models.items():
    vals = {}
    for fam in families:
        mask = (y_test_family == fam)
        vals[fam] = round(float(pred[mask].mean()), 4) if mask.sum() else np.nan
    # False positive rate trên Normal
    vals["Normal(FPR)"] = round(float(pred[y_test_family == "Normal"].mean()), 4)
    recall_by_family[name] = vals

family_table = pd.DataFrame(recall_by_family).T
family_table

,DoS,Probe,R2L,U2R,Normal(FPR)
LogReg (supervised),0.8193,0.7580,0.0125,0.2239,0.0739
RandomForest (supervised),0.7857,0.7720,0.0572,0.1493,0.0271
XGBoost (supervised),0.8509,0.7063,0.0620,0.1791,0.0280
Euclidean (unsupervised),0.3184,0.8658,0.3723,0.5970,0.0758
Mahalanobis (unsupervised),0.9083,0.9075,0.5795,0.7463,0.0740


## Checklist hoàn thành

- [x] Đã huấn luyện ít nhất 2 mô hình supervised.
- [x] Đã so sánh trên cùng test set và cùng metric.
- [x] Có bảng recall theo họ tấn công.
- [x] Có nhận xét về ưu/nhược điểm từng hướng.